In [1]:
import os
import math
import time

import cv2
import numpy as np
import pandas as pd
import requests
from PIL import Image

## Check generated fakes

In [2]:
fake_files = sorted(f for f in os.listdir("data/fakes") if f.endswith(".png"))
print(len(fake_files), fake_files[:3], fake_files[-3:])
print({Image.open(f"data/fakes/{f}").size for f in fake_files})

2800 ['seed0000.png', 'seed0001.png', 'seed0002.png'] ['seed2797.png', 'seed2798.png', 'seed2799.png']
{(512, 512)}


## Download real images

In [3]:
df = pd.read_csv("data/deepfake.csv")
real = df[df["label"] == "REAL"].copy()
real["photo_id"] = real["image_url"].str.extract(r"(photo-[0-9a-f-]+)")[0]
print(len(real), "rows,", real["photo_id"].nunique(), "unique photos")
real = real.drop_duplicates("photo_id")

2790 rows, 2349 unique photos


In [4]:
os.makedirs("data/reals", exist_ok=True)
failures = []
session = requests.Session()

for i, row in enumerate(real.itertuples(), start=1):
    path = f"data/reals/{row.photo_id}.jpg"
    if os.path.exists(path):
        continue
    try:
        resp = session.get(row.image_url, timeout=30)
        resp.raise_for_status()
        content_type = resp.headers.get("Content-Type", "")
        if not content_type.startswith("image/"):
            raise ValueError(f"not an image: {content_type}")
        with open(path, "wb") as f:
            f.write(resp.content)
    except Exception as e:
        failures.append({"photo_id": row.photo_id, "url": row.image_url, "error": str(e)})
    if i % 100 == 0:
        print(f"{i}/{len(real)} done, {len(failures)} failures")
    time.sleep(0.2)

pd.DataFrame(failures).to_csv("data/real_download_failures.csv", index=False)
print("Finished with", len(failures), "failures")

Finished with 0 failures


## Crop and align faces

In [5]:
SIZE = 224          # final image size in pixels
CROP_SCALE = 2.6    # crop width = 2.6 x the distance between the eyes

detector = cv2.FaceDetectorYN.create("models/face_detection_yunet_2023mar.onnx", "", (320, 320), score_threshold=0.9)

def is_grayscale(img):
    b, g, r = [img[..., i].astype(int) for i in range(3)]
    return np.abs(b - g).mean() < 3 and np.abs(g - r).mean() < 3

def align_face(path):
    img = cv2.imread(path)
    h, w = img.shape[:2]

    # 1. Find the face, shrinking the image first because YuNet misses very large faces
    faces = None
    for target in (640, 480, 320):
        scale = min(1.0, target / max(h, w))
        small = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        detector.setInputSize((small.shape[1], small.shape[0]))
        _, faces = detector.detect(small)
        if faces is not None:
            break
    if faces is None:
        return None, "no_face"
    face = max(faces, key=lambda f: f[2] * f[3])[:14] / scale

    # 2. Use the eye and mouth positions to find the crop's center, size and angle
    (lx, ly), (rx, ry) = sorted([face[4:6], face[6:8]], key=lambda p: p[0])
    mouth = (face[10:12] + face[12:14]) / 2
    eye_mid = np.array([(lx + rx) / 2, (ly + ry) / 2])
    center = eye_mid + 0.1 * (mouth - eye_mid)
    side = CROP_SCALE * math.hypot(rx - lx, ry - ly)
    angle = math.degrees(math.atan2(ry - ly, rx - lx))
    if side < SIZE:
        return None, "too_small"

    # 3. Check that the rotated crop stays inside the photo
    M = cv2.getRotationMatrix2D((float(center[0]), float(center[1])), angle, 1.0)
    half = side / 2
    corners = np.array([[center[0] - half, center[1] - half], [center[0] + half, center[1] - half],
                        [center[0] - half, center[1] + half], [center[0] + half, center[1] + half]])
    inv = cv2.invertAffineTransform(M)
    src = corners @ inv[:, :2].T + inv[:, 2]
    if src.min() < 0 or (src[:, 0] > w).any() or (src[:, 1] > h).any():
        return None, "out_of_bounds"

    # 4. Rotate so the eyes are level, crop the square, shrink to SIZE
    rotated = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC)
    x0, y0, s = int(round(center[0] - half)), int(round(center[1] - half)), int(round(side))
    crop = cv2.resize(rotated[y0:y0 + s, x0:x0 + s], (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    if is_grayscale(crop):
        return None, "grayscale"
    return crop, "ok"

[ WARN:0@0.963] global net_impl_backend.cpp:388 setPreferableTarget Targets are not supported by the new graph engine for now


In [6]:
log = []
for kind in ["reals", "fakes"]:
    os.makedirs(f"data/processed/{kind}", exist_ok=True)
    for f in sorted(os.listdir(f"data/{kind}")):
        if not f.endswith((".jpg", ".png")):
            continue
        crop, status = align_face(f"data/{kind}/{f}")
        if crop is not None:
            name = os.path.splitext(f)[0] + ".jpg"
            cv2.imwrite(f"data/processed/{kind}/{name}", crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
        log.append({"kind": kind, "file": f, "status": status})

log = pd.DataFrame(log)
log.to_csv("data/processed/preprocess_log.csv", index=False)
pd.crosstab(log["status"], log["kind"])

kind,fakes,reals
status,,
grayscale,0,145
no_face,0,796
ok,2797,786
out_of_bounds,0,343
too_small,3,279


## Build manifest

In [7]:
reals = sorted(f for f in os.listdir("data/processed/reals") if f.endswith(".jpg"))
fakes = sorted(f for f in os.listdir("data/processed/fakes") if f.endswith(".jpg"))

real_df = pd.DataFrame({"path": [f"data/processed/reals/{f}" for f in reals],
                        "label": "real", "source": "unsplash"})
fake_df = pd.DataFrame({"path": [f"data/processed/fakes/{f}" for f in fakes],
                        "label": "fake", "source": "stylegan3-t-ffhq"})
fake_df = fake_df.sample(n=len(real_df), random_state=42)

manifest = pd.concat([real_df, fake_df], ignore_index=True)
manifest["label_numeric"] = (manifest["label"] == "fake").astype(int)

In [8]:
from sklearn.model_selection import GroupShuffleSplit

manifest = manifest.sort_values("path").reset_index(drop=True)

# Group reals into photo-shoot series: a photo uploaded within 2 s of the previous one joins its series
real = manifest[manifest["label"] == "real"].copy()
real["upload_ms"] = real["path"].str.extract(r"photo-(\d+)-")[0].astype("int64")
real = real.sort_values("upload_ms")
real["series"] = (real["upload_ms"].diff() > 2000).cumsum()

# Split the reals by series, so each series lands entirely in one split
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss.split(real, groups=real["series"]))
temp = real.iloc[temp_idx]
gss = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss.split(temp, groups=temp["series"]))

manifest.loc[real.index[train_idx], "split"] = "train"
manifest.loc[temp.index[val_idx], "split"] = "val"
manifest.loc[temp.index[test_idx], "split"] = "test"

# Give the fakes the same count per split as the reals, so every split stays balanced
counts = manifest.loc[manifest["label"] == "real", "split"].value_counts()
n_train, n_val = counts["train"], counts["val"]
fakes = manifest[manifest["label"] == "fake"].sample(frac=1, random_state=42).index
manifest.loc[fakes[:n_train], "split"] = "train"
manifest.loc[fakes[n_train:n_train + n_val], "split"] = "val"
manifest.loc[fakes[n_train + n_val:], "split"] = "test"

manifest = manifest.sort_values(["split", "label", "path"]).reset_index(drop=True)
manifest.to_csv("manifest.csv", index=False)
pd.crosstab(manifest["split"], manifest["label"])

label,fake,real
split,,
test,119,119
train,560,560
val,107,107


## Dataset construction summary

### Fake images
- Model: StyleGAN3-T pretrained on FFHQ, 1024x1024 (NVIDIA, `stylegan3-t-ffhq-1024x1024.pkl`)
- Settings: truncation 0.7, seeds 0–2799 (2,800 images)
- Generated on Google Colab (T4 GPU). StyleGAN3's custom CUDA plugins failed to compile on current Colab software, so they were disabled and the plain-PyTorch versions used (same images, slower).
- Resized to 512x512 (Lanczos) and saved as PNG. Stored in the shared Drive folder `deepfake_project/fakes_512`.

### Real images
2349 unique Unsplash images downloaded, all JPEG, RGB, 1080px wide.

### Preprocessing (same pipeline for both classes)
- Face detection: OpenCV YuNet (`face_detection_yunet_2023mar.onnx`), tried at 640 / 480 / 320 px; largest face used.
- Alignment: rotated so the eyes are level; square crop centered between the eyes and the mouth, side = 2.6 x the distance between the eyes.
- Dropped if: no face found; crop smaller than 224 px (would need enlarging); crop runs off the photo; image is grayscale (no fakes are grayscale).
- Resized to 224x224 and saved as JPEG quality 90.

The reals became the limiting class, with only 786 suitable reals and 2797 fakes.

### Manifest and split
- 786 fakes randomly sampled to match the reals count.
- Many Unsplash reals come in series from the same photo shoot (196 of 786 were uploaded within 2 s of another). A plain random split put 34 of the 118 test reals next to a same-series photo in train. The final split groups reals by series (upload gap of 2 s or less) using `GroupShuffleSplit`, so no series spans two splits.
- Age and gender labels were dropped due to unreliability.
- Final split: train 560 / 560, val 107 / 107, test 119 / 119 (1,572 images)